# Fine-tune German BERT on Colab (optional)

Runs the repository's own command `jobnlp train-classifier` on a GPU. Inputs come from your private Drive folder `MyDrive/jobnlp/` (see docs/COLAB_RUNBOOK.md). Checkpoints are written to Drive every epoch, so a disconnected session resumes when you run all cells again. Metrics computed here are not used: the local `jobnlp import-colab` re-evaluates the returned model.

In [ ]:
# 1. GPU check
!nvidia-smi || echo 'No GPU: choose Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/jobnlp'

In [ ]:
# 3. Unpack the code and install pinned requirements (CPU torch line is ignored on Colab's GPU torch)
!rm -rf /content/jobnlp && mkdir -p /content/jobnlp
!cd /content/jobnlp && unzip -q $DRIVE/jobnlp_code.zip
!grep -v -E '^(torch==|--extra-index-url)' /content/jobnlp/requirements.txt > /content/req.txt
!pip install -q -r /content/req.txt
!pip install -q --no-deps -e /content/jobnlp

In [ ]:
# 4. Place the inputs and check them against expected.json
import hashlib, json, os, shutil
os.makedirs('/content/jobnlp/data/processed', exist_ok=True)
os.makedirs('/content/jobnlp/results', exist_ok=True)
shutil.copy(f'{DRIVE}/colab_inputs/postings.parquet', '/content/jobnlp/data/processed/postings.parquet')
shutil.copy(f'{DRIVE}/colab_inputs/split_manifest.json', '/content/jobnlp/results/split_manifest.json')
expected = json.load(open(f'{DRIVE}/colab_inputs/expected.json'))
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
assert sha('/content/jobnlp/data/processed/postings.parquet') == expected['corpus_sha256'], 'corpus checksum mismatch'
assert sha('/content/jobnlp/configs/default.yaml') == expected['config_sha256'], 'config checksum mismatch'
os.environ['JOBNLP_GIT_COMMIT'] = expected['git_commit']
print('inputs verified')

In [ ]:
# 5. Train: full training set, distilled and larger model, three seeds. Each run writes
#    checkpoints to Drive every epoch and resumes from them after a disconnect.
RUNS = [(m, s) for m in ['distilbert/distilbert-base-german-cased', 'deepset/gbert-base'] for s in (0, 1, 2)]
for model_id, seed in RUNS:
    out = f"{DRIVE}/outputs/{model_id.split('/')[-1]}_seed{seed}"
    if os.path.exists(f'{out}/model/manifest.json'):
        print('done already:', out); continue
    !cd /content/jobnlp && jobnlp train-classifier --models bert --bert-model {model_id} --seed {seed} --bert-epochs 3 --max-train 1000000 --out-dir {out}

In [ ]:
# 6. What to download: each folder DRIVE/outputs/<model>_seed<k>/model (weights, tokenizer,
#    manifest.json). See docs/COLAB_RUNBOOK.md for where to put them locally.
!ls -la $DRIVE/outputs/*/model | head -50